## 1. Load Raw Data

In [1]:
import pandas as pd

In [2]:
customers = pd.read_csv("../data/raw/Customers.csv", sep=";")
products = pd.read_csv("../data/raw/Products.csv", sep=";")
campaigns = pd.read_csv("../data/raw/Campaigns.csv", sep=";")
transactions = pd.read_csv("../data/raw/Transactions.csv", sep=";")

In [3]:
customers.head()

,customer_id,name,city,country
0,C001,John Doe,New York,USA
1,C002,Jane Smith,Los Angeles,USA
2,C003,Alice Johnson,Chicago,USA
3,C004,Bob Wilson,Houston,USA
4,C005,Emma Brown,Miami,USA


## 2. Initial Data Quality Check
Basic sanity check on each file individually: shape, missing values, duplicate rows.

### Customers

In [4]:
print("shape:", customers.shape)
print("missing values:\n", customers.isnull().sum())
print("duplicate rows:", customers.duplicated().sum())

shape: (50, 4)
missing values:
 customer_id    0
name           0
city           0
country        0
dtype: int64
duplicate rows: 0


### Products

In [5]:
print("shape:", products.shape)
print("missing values:\n", products.isnull().sum())
print("duplicate rows:", products.duplicated().sum())

shape: (20, 6)
missing values:
 product_id        0
name              0
category          0
supplier          0
unit_price        0
stock_quantity    0
dtype: int64
duplicate rows: 0


### Campaigns

In [6]:
print("shape:", campaigns.shape)
print("missing values:\n", campaigns.isnull().sum())
print("duplicate rows:", campaigns.duplicated().sum())

shape: (5, 7)
missing values:
 campaign_id        0
campaign_name      0
start_date         0
end_date           0
channel            0
budget             0
target_audience    0
dtype: int64
duplicate rows: 0


### Transactions

In [7]:
print("shape:", transactions.shape)
print("missing values:\n", transactions.isnull().sum())
print("duplicate rows:", transactions.duplicated().sum())

shape: (3998, 6)
missing values:
 transaction_id    0
date              0
customer_id       0
product_id        0
quantity          0
campaign_id       0
dtype: int64
duplicate rows: 0


## 3. Testing Whether the Tables Merge Cleanly
Attempting to merge transactions with customers and with campaigns, one at a time, to check whether the IDs line up before building anything final.

In [8]:
# Merging transactions with customers on customer_id to check how many rows fail to match

customer_check = transactions.merge(customers, on = "customer_id", how = "left")
customer_check["name"].isnull().sum()

np.int64(3998)

In [9]:
# Merging transactions with campaigns on campaign_id, before fixing the prefix, to confirm the same issue exists here

campaign_check = transactions.merge(campaigns, on = "campaign_id", how = "left")
campaign_check["campaign_name"].isnull().sum()

np.int64(3998)

## 4. Investigating Merge Mismatches
Checking whether every transaction matched cleanly to a customer and a campaign. Some rows didn't, investigating why.

In [10]:
# Comparing customer_id format between transactions and customers to spot why the merge failed

print(transactions["customer_id"].head())
print(customers["customer_id"].head())

0    C1020
1    C1042
2    C1005
3    C1031
4    C1047
Name: customer_id, dtype: object
0    C001
1    C002
2    C003
3    C004
4    C005
Name: customer_id, dtype: object


In [11]:
# Removing extra "1" prefix in customer_id before merging again

transactions["customer_id"] = transactions["customer_id"].str.replace("C1", "C", n=1)

customer_check = transactions.merge(customers, on="customer_id", how="left")
customer_check["name"].isnull().sum()

np.int64(0)

In [12]:
# Removing extra "1" prefix in campaign_id before merging again

transactions["campaign_id"] = transactions["campaign_id"].str.replace("MC1", "MC", n=1)

campaign_check = transactions.merge(campaigns, on="campaign_id", how="left")
campaign_check["campaign_name"].isnull().sum()

np.int64(0)

## 5. Last Fix

- Fixing the date format (currently day/month/year, needs to be read correctly before any seasonal analysis)


In [13]:
# Converting the date column to an actual date type, telling pandas it's day/month/year format

transactions["date"] = pd.to_datetime(transactions["date"], format="%d/%m/%y")
transactions["date"].head()

0   2023-05-04
1   2023-07-23
2   2023-06-30
3   2023-05-29
4   2023-06-12
Name: date, dtype: datetime64[ns]

## 6. Building the Final Merged Dataset
Combining transactions, customers, products, and campaigns into one table now that the ID issues are fixed.

In [14]:
# Final combined table: transactions + customers + products + campaigns

merged = transactions.merge(customers, on = "customer_id", how = "left")
merged = merged.merge(products, on = "product_id", how = "left")
merged = merged.merge(campaigns, on = "campaign_id", how = "left")

merged.shape

(3998, 20)

## 7. Adding a Revenue Column
Revenue isn't in the data directly, it's quantity × unit price, calculated once here so all three insights can reuse it.

In [15]:
# Calculating revenue per transaction: quantity sold times the unit price of that product

merged["revenue"] = merged["quantity"] * merged["unit_price"]
merged["revenue"].head()

0    150
1     25
2    180
3    360
4    720
Name: revenue, dtype: int64

## 8. Insight 1: Store (City) Performance
Using customer city as a stand-in for store/region, since no store-level data exists. Comparing total revenue across the five cities to identify which are over- or under-performing.

In [16]:
# Total revenue by city, sorted highest to lowest
revenue_by_city = merged.groupby("city")["revenue"].sum().sort_values(ascending = False)
revenue_by_city

city
Los Angeles    382865
Houston        325255
New York       271915
Chicago        259565
Miami          225720
Name: revenue, dtype: int64

In [17]:
# Number of customers per city, to see if lower revenue is just due to fewer customers

customers["city"].value_counts()

city
Los Angeles    13
Houston        11
New York        9
Chicago         9
Miami           8
Name: count, dtype: int64

In [18]:
# Revenue per customer, to check if cities are truly under/overperforming or just have fewer customers

customer_counts = customers["city"].value_counts()
revenue_per_customer = revenue_by_city / customer_counts
revenue_per_customer.sort_values(ascending=False)

city
New York       30212.777778
Houston        29568.636364
Los Angeles    29451.153846
Chicago        28840.555556
Miami          28215.000000
dtype: float64

### Finding

Total revenue by city ranking (Los Angeles → Houston → New York → Chicago → Miami) closely mirrors the number of customers per city, not differences in spending behavior. When revenue is divided by customer count, all five cities land within a tight range ($28,215–$30,213 per customer, about a 7% spread).

**Takeaway:** Cities are not underperforming on a per-customer basis. Lower total revenue in cities like Miami and Chicago reflects fewer customers (8 and 9, versus 13 in Los Angeles), not weaker spending per customer. The actionable lever for management is customer acquisition in lower-revenue cities, not "fixing" store/regional performance, since there's no evidence performance itself is the problem.

**Caveat:** City is used as a stand-in for store/region, since no store-level data exists in the provided files.

## 9. Insight 2: Customer Purchasing Patterns
Looking at what customers actually buy: which product categories drive revenue, and how purchase sizes vary, to understand preferences and inform product offering decisions.

In [19]:
# Revenue and quantity sold by product category

merged.groupby("category")[["revenue", "quantity"]].sum().sort_values("revenue", ascending=False)

,revenue,quantity
category,,
Accessories,742650,10981
Electronics,722670,11282


In [20]:
# Revenue by individual product, sorted highest to lowest

# merged.groupby("name")["revenue"].sum().sort_values(ascending = False)

In [21]:
merged.columns

Index(['transaction_id', 'date', 'customer_id', 'product_id', 'quantity',
       'campaign_id', 'name_x', 'city', 'country', 'name_y', 'category',
       'supplier', 'unit_price', 'stock_quantity', 'campaign_name',
       'start_date', 'end_date', 'channel', 'budget', 'target_audience',
       'revenue'],
      dtype='object')

In [22]:
# Renaming columns left over from the merge

merged = merged.rename(columns={"name_x": "customer_name", "name_y": "product_name"})

In [23]:
# Revenue by individual product, sorted highest to lowest

merged.groupby("product_name")["revenue"].sum().sort_values(ascending = False)

product_name
Bluetooth Headphones Charger    182700
Standing Desk                   177660
Briefcase                       155280
Bluetooth Speaker               140140
Keyboard                        122320
Computer Case                   119570
Suitcase                        116880
External Microphone              64500
Gaming Headset                   59250
Keyboard Mousepad                50700
Adapter                          48520
Desk Chair                       41440
Backpack                         31800
Bluetooth Headphones             30420
Smart Watch                      27700
Wireless Mouse                   24220
Cleanup Kit                      23400
Laptop                           22900
USB-C Cable                      15450
Phone Case                       10470
Name: revenue, dtype: int64

In [24]:
# Quantity sold by product, sorted highest to lowest, to compare against the revenue ranking

merged.groupby("product_name")["quantity"].sum().sort_values(ascending = False)

product_name
Briefcase                       1294
Standing Desk                   1269
Bluetooth Headphones Charger    1218
Adapter                         1213
Wireless Mouse                  1211
Gaming Headset                  1185
Cleanup Kit                     1170
Laptop                          1145
Keyboard                        1112
Smart Watch                     1108
Computer Case                   1087
External Microphone             1075
Backpack                        1060
Phone Case                      1047
Desk Chair                      1036
USB-C Cable                     1030
Bluetooth Headphones            1014
Keyboard Mousepad               1014
Bluetooth Speaker               1001
Suitcase                         974
Name: quantity, dtype: int64

In [25]:
# Quantity sold by city and category, to check for regional preference differences

merged.groupby(["city", "category"])["quantity"].sum()

city         category   
Chicago      Accessories    2113
             Electronics    1992
Houston      Accessories    2286
             Electronics    2678
Los Angeles  Accessories    2960
             Electronics    2851
Miami        Accessories    1638
             Electronics    1722
New York     Accessories    1984
             Electronics    2039
Name: quantity, dtype: int64

In [26]:
# Percentage split between Accessories and Electronics, per city

city_category = merged.groupby(["city", "category"])["quantity"].sum()
city_category_pct = city_category / city_category.groupby(level = 0).sum() * 100
city_category_pct

city         category   
Chicago      Accessories    51.473812
             Electronics    48.526188
Houston      Accessories    46.051571
             Electronics    53.948429
Los Angeles  Accessories    50.937876
             Electronics    49.062124
Miami        Accessories    48.750000
             Electronics    51.250000
New York     Accessories    49.316431
             Electronics    50.683569
Name: quantity, dtype: float64

### Finding NO WHALE CUSTOMER !!!!!

Product-level purchase quantities are remarkably even across the catalog (974–1,294 units per product), meaning there's no strong customer preference for specific products by volume. The revenue leaderboard is driven almost entirely by price, not demand, a $150 item and a $10 item sell in nearly identical quantities.

Category preference (Electronics vs Accessories) is also close to a 50/50 split in most cities. The one exception is Houston, which leans noticeably toward Electronics (54% vs 46%), while every other city stays within a couple points of even. 

**Takeaway:** There's little evidence of strong product-level demand differences to act on broadly, but Houston's electronics lean is a specific, real signal worth a follow-up (e.g., stocking or marketing emphasis on electronics specifically for that market).

**Caveat:** City is used as a stand-in for store/region, since no store-level data exists in the provided files.

## 10. Insight 3: Seasonal Trends
Looking at how revenue moves across the year, independent of campaigns, to spot busy and slow periods.

In [27]:
# Revenue by month, across the full year

merged["month"] = merged["date"].dt.month
revenue_by_month = merged.groupby("month")["revenue"].sum()
revenue_by_month

month
1     135940
2     103330
3     122615
4     123290
5     130950
6     127045
7     116885
8     114535
9     120015
10    121965
11    136990
12    111760
Name: revenue, dtype: int64

In [28]:
# Checking how many different years exist in the date range
merged["date"].dt.year.value_counts()

date
2023    3982
2024      16
Name: count, dtype: int64

In [29]:
# Revenue for transactions that happened specifically during the Black Friday campaign window

black_friday = merged[merged["campaign_name"] == "Black Friday"]
black_friday["revenue"].sum()

np.int64(280550)

In [30]:
# Checking what months the Black Friday-tagged transactions actually happened in

black_friday["month"].value_counts()

month
3     83
1     81
11    74
10    72
6     63
5     61
12    60
9     58
8     56
2     55
4     53
7     53
Name: count, dtype: int64

In [31]:
# Revenue by month, broken down by category

merged.groupby(["month", "category"])["revenue"].sum()

month  category   
1      Accessories    61490
       Electronics    74450
2      Accessories    54720
       Electronics    48610
3      Accessories    56750
       Electronics    65865
4      Accessories    60660
       Electronics    62630
5      Accessories    64905
       Electronics    66045
6      Accessories    73665
       Electronics    53380
7      Accessories    54150
       Electronics    62735
8      Accessories    61675
       Electronics    52860
9      Accessories    67855
       Electronics    52160
10     Accessories    56155
       Electronics    65810
11     Accessories    65645
       Electronics    71345
12     Accessories    64980
       Electronics    46780
Name: revenue, dtype: int64

### Finding

Revenue by month shows two clear standouts: November ($136,990) and January ($135,940) as the highest months, with February ($103,330) as the lowest and December ($111,760) surprisingly on the low end, not the holiday spike a retail chain might be expected to show.

This pattern doesn't hold evenly across both categories. It's driven mainly by Electronics, which swings widely across the year (from $46,780 in December, its lowest month, up to $74,450 in January), while Accessories stays comparatively steady all year (ranging roughly $54,150–$73,665, with no December dip at all).

Checked whether campaign activity explains the November spike specifically, it doesn't. Transactions tagged "Black Friday" turned out to be spread almost evenly across all twelve months (53–83 per month), not clustered in November as expected. This means campaign_id labels in this dataset don't reliably reflect when a campaign actually ran, so campaign timing can't be used to explain seasonal spikes, and reinforces the earlier caveat that marketing effectiveness (concern #3) can't be reliably assessed with this data.

**Takeaway:** The seasonal pattern is real but concentrated in Electronics specifically, not a company-wide effect. Worth investigating why Electronics drops sharply in December (a genuine business question worth raising, not something the data explains on its own), and worth flagging to the Data team that campaign attribution in the source data doesn't line up with real campaign dates.

**Verification note:** Confirmed the January spike isn't a data artifact — 2024 accounts for less than 1% of transactions, so it's not double-counting two different Januaries.

In [32]:
# Exporting final merged dataset for Power BI

merged.to_csv("../data/clean/merged_retailco.csv", index=False)